In [0]:
%run ./00_config

In [0]:
# Ensure schema exists
# -----------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS h2.ops")
# -----------------------------
# Create table if missing
# -----------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_PIPELINE_LOG} (
  run_ts_utc TIMESTAMP,
  run_id STRING,
  stage STRING,
  status STRING,
  row_count BIGINT
) USING DELTA
""")

In [0]:
current_cols = [r.col_name for r in spark.sql(f"DESCRIBE TABLE {T_PIPELINE_LOG}").collect()]
required_cols = ["run_ts_utc", "run_id", "stage", "status", "row_count"]
if "run_id" not in current_cols:
    spark.sql(f"ALTER TABLE {T_PIPELINE_LOG} ADD COLUMN run_id STRING")
    current_cols = [r.col_name for r in spark.sql(f"DESCRIBE TABLE {T_PIPELINE_LOG}").collect()]

In [0]:
row_count = 0  # Define row_count variable; set to appropriate value as needed

if "run_id" in current_cols:
    spark.sql(f"""
    INSERT INTO {T_PIPELINE_LOG} (run_ts_utc, run_id, stage, status, row_count)
    SELECT
      current_timestamp(),
      '{RUN_ID}',
      'full_pipeline',
      'success',
      CAST({row_count} AS BIGINT)
    """)
else:
    spark.sql(f"""
    INSERT INTO {T_PIPELINE_LOG} (run_ts_utc, stage, status, row_count)
    SELECT
      current_timestamp(),
      'full_pipeline',
      'success',
      CAST({row_count} AS BIGINT)
    """)

In [0]:
display(
    spark.sql(f"""
    SELECT *
    FROM {T_PIPELINE_LOG}
    ORDER BY run_ts_utc DESC
    LIMIT 20
    """)
)